# 09 -- Manufacturing Plan

Composite layup plan, structural mass estimation, materials shopping list,
and step-by-step manufacturing procedure for the BWB UAV.

Prerequisites: design selected from catalog (notebook 06), validated (notebook 07).


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np

from src.parameterization.design_variables import BWBParams, params_from_vector

try:
    best_x = np.load('../output/best_x_v2.npy')
    params = params_from_vector(best_x)
    print('Loaded v2 optimized design')
except FileNotFoundError:
    params = BWBParams()
    print('Using default params')

from src.visualization.style import apply_style, COLORS
apply_style()


In [ ]:
# 5. Composite Layup Plan
from src.manufacturing.composite import ManufacturingPlan

plan = ManufacturingPlan.default_bwb(params)
plan.print_layup_summary()

In [ ]:
# 6. Structural Mass: Layup vs Empirical
plan.print_mass_breakdown()

In [ ]:
# 7. Materials Shopping List
from src.systems.avionics import AvionicsSpec
avionics = AvionicsSpec.default_bwb()

plan.print_materials_bom()

print(f'Avionics cost:  {avionics.total_price_eur:.0f}\u20ac')
bom = plan.compute_materials_bom()
mat_cost = sum(item['price_eur'] for item in bom)
print(f'Materials cost:  {mat_cost:.0f}\u20ac')
print(f'TOTAL estimate: {avionics.total_price_eur + mat_cost:.0f}\u20ac (excl. tooling, consumables)')

In [ ]:
# 8. Manufacturing Procedure
from src.manufacturing.composite import print_manufacturing_steps
print_manufacturing_steps()

In [ ]:
# 5. Mass & Cost Visualization
import matplotlib.pyplot as plt
from src.visualization.style import COLORS

zones = plan.estimate_composite_mass(params)
total_layup = zones.pop('total_kg')
fittings = zones.pop('fittings_glue_finish')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Bar chart: structural mass by zone (layup vs empirical) ---
zone_names = list(zones.keys())
zone_masses = [v * 1000 for v in zones.values()]  # kg -> g
zone_colors = [COLORS['body'] if 'body' in z else
               COLORS['wing'] if 'wing' in z or 'spar' in z else
               COLORS['elevon'] if 'control' in z or 'leading' in z else
               COLORS['secondary'] for z in zone_names]

bars = ax1.barh(zone_names, zone_masses, color=zone_colors, edgecolor='white')
ax1.barh(['Fittings/glue'], [fittings * 1000], color=COLORS['light'], edgecolor='white')

from src.parameterization.bwb_aircraft import estimate_structural_mass
emp = estimate_structural_mass(params) * 1000
ax1.axvline(emp, color=COLORS['infeasible'], linestyle='--', linewidth=2,
            label=f'Empirical total: {emp:.0f}g')
ax1.axvline(total_layup * 1000, color=COLORS['feasible'], linestyle='-', linewidth=2,
            label=f'Layup total: {total_layup*1000:.0f}g')
ax1.set_xlabel('Mass [g]')
ax1.set_title('Structural Mass by Zone', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.invert_yaxis()

# --- Pie chart: total cost breakdown ---
mat_cost = sum(item['price_eur'] for item in plan.compute_materials_bom())
avio_cost = avionics.total_price_eur
tooling_est = 50  # rough estimate for MDF + CNC time

cost_labels = ['Composites', 'Avionics', 'Tooling (est.)']
cost_values = [mat_cost, avio_cost, tooling_est]
cost_colors = [COLORS['body'], COLORS['avionics'], COLORS['secondary']]

ax2.pie(cost_values, labels=cost_labels, colors=cost_colors,
        autopct=lambda p: f'{p*sum(cost_values)/100:.0f}\u20ac\n({p:.0f}%)',
        startangle=90, textprops={'fontsize': 10})
ax2.set_title(f'Cost Breakdown (total ~{sum(cost_values):.0f}\u20ac)',
              fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../output/manufacturing_mass_cost.png', dpi=150, bbox_inches='tight')
plt.show()